[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/090.%20The%20Global%20Sourcing%20Offshoring%20vs.%20Reshoring%20Problem/P90-Tier-8_executed.ipynb)

---


# 90. The Global Sourcing Offshoring vs. Reshoring Problem
## Tier 8: A Value-Aligned Ethical Sourcing Framework

### Purpose and scope

This notebook demonstrates a sourcing decision process that screens suppliers against selected ethical thresholds, generates candidate sourcing scenarios, scores those scenarios against constitutional principles and stakeholder utilities, and selects the candidate with the highest combined ethical score. It also produces a transparency report, a bias-risk analysis, summary metrics, and visualizations.

The method used here is **random candidate-scenario generation followed by weighted scoring and selection**. It is an illustrative decision framework, not a mathematical-programming solver and not evidence that the selected sourcing plan is globally optimal. Its result depends on the supplier data, the specified weights, the screening rules, and the scenarios generated with the fixed random seed.

### Inputs and outputs

**Inputs** are the supplier records, ethical screening thresholds, principle and stakeholder weights, product demands, and the problem context. **Outputs** are the generated scenarios, their scores, the highest-scoring scenario, a report, bias indicators, and plots. The supplier database contains example data for teaching; it is not an independently verified audit dataset.

### Modeling notes

- Supplier screening applies the wage, safety, carbon-intensity, renewable-energy, and transparency thresholds defined in the code. Other ethical requirements in the context are recorded for discussion but are not enforced unless represented by a coded threshold.
- Candidate allocations divide each product's demand among the selected suppliers in proportion to their combined ethical, environmental, and social scores. Supplier capacity is recorded but is not imposed as an allocation constraint. Thus, allocations illustrate the scoring method and should not be treated as validated purchase orders.
- Stakeholder utilities are constructed as normalized illustrative scores. The weighted score is a decision aid, not a directly measured ethical truth or a substitute for stakeholder consultation.
- Relationship and cognitive bias indicators are simulated random values. They are not inferred from evidence about people or suppliers. A detected-bias label indicates that an indicator exceeds the chosen threshold; the code does not modify the selected allocation to mitigate that bias.
- Business-impact figures are not supplied as empirical results. The metrics section reports quantities that can be calculated from the example inputs and identifies any proxy assumptions explicitly.

### Method and mathematical representation

Let \(S\) be the set of suppliers that pass the screening rules, and let \(\mathcal{C}\) be the set of candidate scenarios generated from subsets of \(S\). Each scenario \(c\in\mathcal{C}\) specifies a supplier subset and an allocation \(q_{pcj}\) of product \(p\)'s demand to supplier \(j\). For each constitutional principle \(r\), let \(a_{cr}\in[0,1]\) be its compliance score and \(\alpha_r\) its weight. For each stakeholder group \(h\), let \(u_{ch}\in[0,1]\) be its utility score and \(\beta_h\) its weight.

The scenario score and selection rule are

$$
A_c=\sum_r \alpha_r a_{cr},
\qquad
H_c=\sum_h \beta_h u_{ch},
\qquad
E_c=0.6A_c+0.4H_c,
\qquad
c^*\in\operatorname*{arg\,max}_{c\in\mathcal{C}} E_c.
$$

Here, \(A_c\) is weighted constitutional compliance, \(H_c\) is weighted stakeholder utility, and \(E_c\) is the combined ethical score. The weights \(\alpha_r\) and \(\beta_h\) each sum to one. If multiple scenarios tie for the highest score, the selection is not unique; the code uses the first maximum in scenario order.

For product \(p\), demand \(D_p\), selected supplier set \(S_c\), and supplier score \(g_{cj}\), the allocation rule is

$$
q_{pcj}=D_p\frac{g_{cj}}{\sum_{k\in S_c}g_{ck}},
\qquad
\sum_{j\in S_c}q_{pcj}=D_p.
$$

In the code, `g_cj` is calculated from each selected supplier's ethical, environmental, and social scores. The allocation is therefore proportional to this combined score; it is not the solution of a capacity-constrained optimization model.

### Learning sequence

The notebook first defines the scoring framework and example data. It then screens suppliers, generates and scores candidate scenarios, selects the highest-scoring candidate, and examines the result through reporting, bias indicators, metrics, and plots. This sequence makes each result traceable to its inputs and scoring rules.

In [ ]:
from collections import Counter
from datetime import datetime
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Fixed seeds make scenario sampling and simulated bias indicators reproducible.
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Define the ethical scoring framework

The framework has four linked parts: constitutional principles describe the values to assess; stakeholder groups describe whose interests enter the score; screening thresholds exclude suppliers that fail selected minimum requirements; and scenario scoring combines the remaining assessments. A score is meaningful only relative to these definitions and weights, so they are displayed explicitly rather than treated as universal standards.

For each scenario, the implementation first computes a score for every principle and stakeholder group. It then takes the weighted sums and combines them using the stated 60% constitutional and 40% stakeholder split. The code's `_evaluate_ethical_scenario` method implements these equations.

In [ ]:
class ConstitutionalAI:
    """Score sourcing scenarios against declared principles and stakeholders."""

    def __init__(self, constitution_name="GlobalTech Ethical Sourcing Constitution"):
        self.constitution_name = constitution_name
        self.constitutional_principles = self._define_constitutional_principles()
        self.stakeholder_registry = self._create_stakeholder_registry()
        self.ethical_constraints = self._define_ethical_constraints()
        self.bias_detector = BiasDetector()
        self.transparency_reporter = TransparencyReporter()

    def _define_constitutional_principles(self):
        """Return principles and weights used in constitutional scoring."""
        return {
            "human_rights": {
                "description": "Respect for human dignity and fundamental rights",
                "weight": 0.25,
                "constraints": ["no_child_labor", "no_forced_labor", "fair_wages", "safe_working_conditions"],
            },
            "environmental_protection": {
                "description": "Protection of the environment and sustainable practices",
                "weight": 0.25,
                "constraints": ["carbon_neutral", "waste_reduction", "renewable_energy", "biodiversity_protection"],
            },
            "fair_competition": {
                "description": "Fair competition and market integrity",
                "weight": 0.20,
                "constraints": ["no_corruption", "fair_pricing", "intellectual_property_respect", "market_transparency"],
            },
            "social_responsibility": {
                "description": "Social responsibility and community development",
                "weight": 0.20,
                "constraints": ["community_investment", "local_development", "education_support", "healthcare_access"],
            },
            "transparency_accountability": {
                "description": "Transparency and accountability in operations",
                "weight": 0.10,
                "constraints": ["full_disclosure", "audit_trail", "stakeholder_communication", "performance_reporting"],
            },
        }

    def _create_stakeholder_registry(self):
        """Return stakeholder groups, normalized weights, and utility functions."""
        return {
            "workers": {
                "description": "Direct and indirect workers in the supply chain",
                "weight": 0.25,
                "utility_function": self._worker_utility,
                "concerns": ["wages", "safety", "job security", "working conditions"],
            },
            "communities": {
                "description": "Local communities affected by operations",
                "weight": 0.20,
                "utility_function": self._community_utility,
                "concerns": ["environment", "jobs", "development", "health"],
            },
            "customers": {
                "description": "End customers and consumers",
                "weight": 0.20,
                "utility_function": self._customer_utility,
                "concerns": ["price", "quality", "ethics", "sustainability"],
            },
            "shareholders": {
                "description": "Company shareholders and investors",
                "weight": 0.15,
                "utility_function": self._shareholder_utility,
                "concerns": ["profitability", "risk", "growth", "reputation"],
            },
            "environment": {
                "description": "Natural environments and ecosystems",
                "weight": 0.10,
                "utility_function": self._environment_utility,
                "concerns": ["climate", "biodiversity", "pollution", "resources"],
            },
            "suppliers": {
                "description": "Supply-chain partners and suppliers",
                "weight": 0.10,
                "utility_function": self._supplier_utility,
                "concerns": ["fair payment", "partnership", "capacity development", "risk sharing"],
            },
        }

    def _define_ethical_constraints(self):
        """Return the explicit supplier-screening thresholds."""
        return {
            "labor_standards": {
                "minimum_wage_multiplier": 2.0,
                "maximum_working_hours": 48,
                "minimum_age": 18,
                "safety_score_minimum": 0.85,
            },
            "environmental_standards": {
                "carbon_intensity_maximum": 0.5,
                "renewable_energy_minimum": 0.3,
                "waste_recycling_minimum": 0.8,
                "water_usage_maximum": 100,
            },
            "business_ethics": {
                "anti_corruption_compliance": 1.0,
                "transparency_score_minimum": 0.8,
                "audit_frequency_minimum": 2,
                "stakeholder_engagement_minimum": 0.7,
            },
        }

    @staticmethod
    def _supplier_quantities(scenario):
        """Aggregate product-by-supplier allocations into supplier quantities."""
        totals = Counter()
        for product_allocation in scenario["allocation"].values():
            totals.update(product_allocation)
        return totals

    def _worker_utility(self, scenario, supplier_data):
        quantities = self._supplier_quantities(scenario)
        total_quantity = sum(quantities.values())
        if total_quantity == 0:
            return 0.0
        weighted_score = sum(
            quantities[supplier]
            * (0.4 * min(supplier_data[supplier].get("wage_multiplier", 1.0) / 3.0, 1.0)
               + 0.4 * supplier_data[supplier].get("safety_score", 0.8)
               + 0.2 * supplier_data[supplier].get("working_conditions", 0.7))
            for supplier in quantities
        ) / total_quantity
        return float(np.clip(weighted_score, 0.0, 1.0))

    def _community_utility(self, scenario, supplier_data):
        quantities = self._supplier_quantities(scenario)
        total_quantity = sum(quantities.values())
        if total_quantity == 0:
            return 0.0
        score = sum(
            quantities[supplier]
            * (0.4 * supplier_data[supplier].get("environmental_score", 0.7)
               + 0.3 * supplier_data[supplier].get("community_investment", 0.5)
               + 0.3 * supplier_data[supplier].get("local_development", 0.6))
            for supplier in quantities
        ) / total_quantity
        return float(np.clip(score, 0.0, 1.0))

    def _customer_utility(self, scenario, supplier_data):
        quantities = self._supplier_quantities(scenario)
        total_quantity = sum(quantities.values())
        if total_quantity == 0:
            return 0.0
        score = sum(
            quantities[supplier]
            * (0.4 * supplier_data[supplier].get("quality", 0.8)
               + 0.3 * supplier_data[supplier].get("ethical_score", 0.7)
               + 0.3 * supplier_data[supplier].get("sustainability", supplier_data[supplier].get("environmental_score", 0.7)))
            for supplier in quantities
        ) / total_quantity
        return float(np.clip(score, 0.0, 1.0))

    def _shareholder_utility(self, scenario, supplier_data):
        average_cost = scenario.get("total_cost", 0.0) / max(scenario.get("total_quantity", 1.0), 1.0)
        cost_utility = 1.0 / (1.0 + average_cost / 250.0)
        risk_utility = 1.0 - scenario.get("risk_score", 0.5)
        brand_utility = scenario.get("brand_score", 0.7)
        return float(np.clip(0.4 * cost_utility + 0.3 * risk_utility + 0.3 * brand_utility, 0.0, 1.0))

    def _environment_utility(self, scenario, supplier_data):
        quantities = self._supplier_quantities(scenario)
        total_quantity = sum(quantities.values())
        if total_quantity == 0:
            return 0.0
        score = sum(
            quantities[supplier]
            * (0.4 / (1.0 + supplier_data[supplier].get("carbon_intensity", 1.0))
               + 0.3 * supplier_data[supplier].get("renewable_energy", 0.2)
               + 0.3 * supplier_data[supplier].get("waste_recycling", 0.6))
            for supplier in quantities
        ) / total_quantity
        return float(np.clip(score, 0.0, 1.0))

    def _supplier_utility(self, scenario, supplier_data):
        quantities = self._supplier_quantities(scenario)
        total_quantity = sum(quantities.values())
        if total_quantity == 0:
            return 0.0
        score = sum(
            quantities[supplier]
            * (0.4 * supplier_data[supplier].get("fair_payment", 0.8)
               + 0.3 * supplier_data[supplier].get("partnership_score", 0.7)
               + 0.3 * supplier_data[supplier].get("capacity_development", 0.6))
            for supplier in quantities
        ) / total_quantity
        return float(np.clip(score, 0.0, 1.0))

    def optimize_ethical_sourcing(self, problem_context, supplier_database):
        """Generate, score, and select the highest-scoring candidate scenario."""
        scenarios = self._generate_ethical_scenarios(problem_context, supplier_database)
        if not scenarios:
            raise ValueError("No candidate scenarios could be generated from eligible suppliers.")
        evaluated = [self._evaluate_ethical_scenario(s, supplier_database) for s in scenarios]
        best = max(evaluated, key=lambda scenario: scenario["ethical_score"])
        report = self.transparency_reporter.generate_report(
            best, self.constitutional_principles, self.stakeholder_registry
        )
        bias = self.bias_detector.analyze_biases(best)
        return {
            "ethical_scenario": best,
            "transparency_report": report,
            "bias_analysis": bias,
            "constitutional_compliance": self._calculate_compliance(best),
            "stakeholder_satisfaction": self._calculate_stakeholder_satisfaction(best),
        }

    def _generate_ethical_scenarios(self, context, supplier_db):
        """Sample candidate supplier subsets and allocate demand by combined scores."""
        eligible = self._filter_ethical_suppliers(supplier_db)
        if len(eligible) < 3:
            raise ValueError("At least three suppliers must pass the screening rules.")

        scenarios = []
        supplier_ids = list(eligible)
        for scenario_id in range(20):
            subset_size = int(np.random.randint(3, min(5, len(supplier_ids)) + 1))
            selected = list(np.random.choice(supplier_ids, size=subset_size, replace=False))
            allocation = self._create_ethical_allocation(selected, eligible)
            total_quantity = sum(sum(product.values()) for product in allocation.values())
            total_cost = sum(
                eligible[supplier]["cost"] * quantity
                for product in allocation.values()
                for supplier, quantity in product.items()
            )
            scenarios.append({
                "id": scenario_id,
                "suppliers": {supplier: eligible[supplier] for supplier in selected},
                "allocation": allocation,
                "context": context,
                "total_quantity": total_quantity,
                "total_cost": total_cost,
                "risk_score": float(np.mean([1.0 - eligible[s]["ethical_score"] for s in selected])),
                "brand_score": float(np.mean([eligible[s]["ethical_score"] for s in selected])),
            })
        return scenarios

    def _filter_ethical_suppliers(self, supplier_db):
        """Keep suppliers satisfying every explicitly coded screening threshold."""
        thresholds = self.ethical_constraints
        eligible = {}
        for supplier_id, data in supplier_db.items():
            passes = (
                data.get("wage_multiplier", 1.0) >= thresholds["labor_standards"]["minimum_wage_multiplier"]
                and data.get("safety_score", 0.8) >= thresholds["labor_standards"]["safety_score_minimum"]
                and data.get("carbon_intensity", 1.0) <= thresholds["environmental_standards"]["carbon_intensity_maximum"]
                and data.get("renewable_energy", 0.2) >= thresholds["environmental_standards"]["renewable_energy_minimum"]
                and data.get("transparency_score", 0.7) >= thresholds["business_ethics"]["transparency_score_minimum"]
            )
            if passes:
                eligible[supplier_id] = data
        return eligible

    def _create_ethical_allocation(self, suppliers, supplier_db):
        """Allocate each product's demand in proportion to combined supplier scores."""
        demands = {"Battery_Tech": 12000, "Solar_Panel": 8000, "Wind_Turbine": 2000}
        scores = {
            supplier: (
                0.4 * supplier_db[supplier].get("ethical_score", 0.7)
                + 0.3 * supplier_db[supplier].get("environmental_score", 0.7)
                + 0.3 * supplier_db[supplier].get("social_score", 0.7)
            )
            for supplier in suppliers
        }
        total_score = sum(scores.values())
        if total_score <= 0:
            raise ValueError("Combined supplier scores must have a positive sum.")
        return {
            product: {supplier: demand * scores[supplier] / total_score for supplier in suppliers}
            for product, demand in demands.items()
        }

    def _evaluate_ethical_scenario(self, scenario, supplier_db):
        """Compute weighted constitutional, stakeholder, and combined scores."""
        principle_scores = {
            principle: self._calculate_principle_compliance(scenario, principle, supplier_db)
            for principle in self.constitutional_principles
        }
        stakeholder_scores = {
            stakeholder: data["utility_function"](scenario, scenario["suppliers"])
            for stakeholder, data in self.stakeholder_registry.items()
        }
        constitutional_score = sum(
            principle_scores[name] * data["weight"]
            for name, data in self.constitutional_principles.items()
        )
        stakeholder_score = sum(
            stakeholder_scores[name] * data["weight"]
            for name, data in self.stakeholder_registry.items()
        )
        evaluation = dict(scenario)
        evaluation.update({
            "constitutional_scores": principle_scores,
            "stakeholder_utilities": stakeholder_scores,
            "constitutional_score": constitutional_score,
            "stakeholder_score": stakeholder_score,
            "ethical_score": 0.6 * constitutional_score + 0.4 * stakeholder_score,
        })
        return evaluation

    def _calculate_principle_compliance(self, scenario, principle, supplier_db):
        """Return a demand-weighted compliance score for one principle."""
        quantities = self._supplier_quantities(scenario)
        total_quantity = sum(quantities.values())
        if total_quantity == 0:
            return 0.0

        def supplier_score(supplier, components):
            data = supplier_db[supplier]
            return sum(weight * data.get(key, default) for key, weight, default in components)

        components = {
            "human_rights": [
                ("wage_multiplier", 0.3, 1.0),
                ("safety_score", 0.4, 0.8),
                ("working_conditions", 0.3, 0.7),
            ],
            "environmental_protection": [
                ("_carbon_inverse", 0.3, 0.5),
                ("renewable_energy", 0.3, 0.2),
                ("waste_recycling", 0.2, 0.6),
                ("biodiversity_protection", 0.2, 0.5),
            ],
            "fair_competition": [
                ("anti_corruption", 0.4, 0.9),
                ("transparency_score", 0.3, 0.7),
                ("fair_pricing", 0.3, 0.8),
            ],
            "social_responsibility": [
                ("community_investment", 0.3, 0.5),
                ("local_development", 0.3, 0.6),
                ("education_support", 0.2, 0.4),
                ("healthcare_access", 0.2, 0.5),
            ],
            "transparency_accountability": [
                ("transparency_score", 0.4, 0.7),
                ("audit_frequency", 0.3, 2.0),
                ("stakeholder_engagement", 0.3, 0.6),
            ],
        }
        if principle not in components:
            return 0.0

        total = 0.0
        for supplier, quantity in quantities.items():
            data = supplier_db[supplier]
            score = 0.0
            for key, weight, default in components[principle]:
                if key == "_carbon_inverse":
                    value = 1.0 / (1.0 + data.get("carbon_intensity", 1.0))
                elif key == "audit_frequency":
                    value = min(data.get(key, default) / 4.0, 1.0)
                elif key == "wage_multiplier":
                    value = min(data.get(key, default) / 3.0, 1.0)
                else:
                    value = data.get(key, default)
                score += weight * value
            total += quantity * score
        return float(np.clip(total / total_quantity, 0.0, 1.0))

    @staticmethod
    def _calculate_compliance(scenario):
        return scenario["constitutional_score"]

    @staticmethod
    def _calculate_stakeholder_satisfaction(scenario):
        return scenario["stakeholder_score"]


class BiasDetector:
    """Calculate descriptive bias indicators for a selected scenario."""

    def __init__(self):
        self.bias_types = ["geographic_bias", "cost_bias", "size_bias", "relationship_bias", "cognitive_bias"]
        self.bias_history = []

    def analyze_biases(self, scenario):
        scores = {kind: self._calculate_bias_score(scenario, kind) for kind in self.bias_types}
        detected = [kind for kind, score in scores.items() if score > 0.3]
        strategies = [self._generate_mitigation_strategy(kind) for kind in detected]
        return {
            "detected_biases": detected,
            "bias_scores": scores,
            "mitigation_strategies": strategies,
            "overall_bias_risk": float(np.mean(list(scores.values()))),
        }

    def _calculate_bias_score(self, scenario, bias_type):
        suppliers = list(scenario["suppliers"].values())
        if bias_type == "geographic_bias":
            locations = [supplier.get("location", "Unknown") for supplier in suppliers]
            return max(Counter(locations).values()) / len(locations)
        if bias_type == "cost_bias":
            costs = [supplier.get("cost", 200) for supplier in suppliers]
            return float(min(np.var(costs) / 10000.0, 1.0))
        if bias_type == "size_bias":
            capacities = [supplier.get("capacity", 30000) for supplier in suppliers]
            return float(min(np.var(capacities) / 1_000_000_000.0, 1.0))
        if bias_type == "relationship_bias":
            return float(np.random.uniform(0.1, 0.4))
        if bias_type == "cognitive_bias":
            return float(np.random.uniform(0.2, 0.5))
        return 0.0

    @staticmethod
    def _generate_mitigation_strategy(bias_type):
        strategies = {
            "geographic_bias": "Review geographic representation and document regional criteria.",
            "cost_bias": "Review cost criteria alongside the declared ethical and stakeholder scores.",
            "size_bias": "Check whether supplier-size differences affect access to consideration.",
            "relationship_bias": "Use documented criteria and independent review of supplier evidence.",
            "cognitive_bias": "Use structured review and record the rationale for each assessment.",
        }
        return strategies.get(bias_type, "Review the decision criteria and supporting evidence.")


class TransparencyReporter:
    """Create a report whose figures are drawn from the evaluated scenario."""

    def __init__(self):
        self.report_sections = [
            "executive_summary", "constitutional_compliance", "stakeholder_impact",
            "ethical_performance", "bias_analysis", "recommendations"
        ]

    def generate_report(self, scenario, principles, stakeholders):
        return {
            "report_date": datetime.now().strftime("%Y-%m-%d"),
            "scenario_id": scenario["id"],
            "sections": {
                section: self._generate_section(section, scenario, principles, stakeholders)
                for section in self.report_sections
            },
        }

    def _generate_section(self, section, scenario, principles, stakeholders):
        if section == "executive_summary":
            content = (
                f"Scenario ID: {scenario['id']}\n"
                f"Suppliers considered: {len(scenario['suppliers'])}\n"
                f"Weighted constitutional score: {scenario['constitutional_score']:.1%}\n"
                f"Weighted stakeholder score: {scenario['stakeholder_score']:.1%}\n"
                f"Combined ethical score: {scenario['ethical_score']:.1%}\n"
                "Interpretation: these scores summarize the stated illustrative model, not an external audit."
            )
            return {"title": "Executive Summary", "content": content}
        if section == "constitutional_compliance":
            lines = [
                f"{name.replace('_', ' ').title()}: {scenario['constitutional_scores'][name]:.1%} "
                f"(weight {data['weight']:.1%}) — {data['description']}"
                for name, data in principles.items()
            ]
            return {"title": "Constitutional Compliance Analysis", "content": "\n".join(lines)}
        if section == "stakeholder_impact":
            lines = [
                f"{name.title()}: {scenario['stakeholder_utilities'][name]:.1%} "
                f"(weight {data['weight']:.1%}) — {data['description']}"
                for name, data in stakeholders.items()
            ]
            return {"title": "Stakeholder Impact Assessment", "content": "\n".join(lines)}
        if section == "ethical_performance":
            content = (
                f"Weighted constitutional score: {scenario['constitutional_score']:.1%}\n"
                f"Weighted stakeholder score: {scenario['stakeholder_score']:.1%}\n"
                f"Combined score (0.6 constitutional + 0.4 stakeholder): {scenario['ethical_score']:.1%}"
            )
            return {"title": "Ethical Performance Metrics", "content": content}
        if section == "bias_analysis":
            return {
                "title": "Bias Analysis and Mitigation",
                "content": "Bias indicators are calculated separately in the bias-analysis step; simulated indicators are not audit findings.",
            }
        if section == "recommendations":
            return {
                "title": "Recommendations and Next Steps",
                "content": (
                    "Validate supplier evidence, consult affected stakeholders, and review the screening thresholds. "
                    "Check capacities and other operational requirements before using an allocation in practice. "
                    "Treat simulated bias indicators as prompts for review, not as proof of bias or its mitigation."
                ),
            }
        return {"title": section, "content": "Content not available."}

## 2. Initialize the framework

The principle weights and stakeholder weights represent the relative importance assigned in this example. Each set sums to one, so each weighted score remains on a comparable 0-to-1 scale when its component scores are also in that range. The combined score then assigns 60% importance to constitutional compliance and 40% to stakeholder utility.

In [ ]:
ethical_ai = ConstitutionalAI("GlobalTech Ethical Sourcing Constitution")

principle_weight_sum = sum(
    item["weight"] for item in ethical_ai.constitutional_principles.values()
)
stakeholder_weight_sum = sum(
    item["weight"] for item in ethical_ai.stakeholder_registry.values()
)
assert np.isclose(principle_weight_sum, 1.0)
assert np.isclose(stakeholder_weight_sum, 1.0)

print("Constitution:", ethical_ai.constitution_name)
print("Constitutional principles:")
for name, data in ethical_ai.constitutional_principles.items():
    print(f"  {name.replace('_', ' ').title()}: weight {data['weight']:.0%}")
print("Stakeholder groups:")
for name, data in ethical_ai.stakeholder_registry.items():
    print(f"  {name.title()}: weight {data['weight']:.0%}")
print("Supplier-screening categories:", ", ".join(ethical_ai.ethical_constraints))

## 3. Provide the supplier data

Each supplier record includes cost and capacity as well as scores and measurements used by the model. The ethical screening step uses only the explicitly coded thresholds. The other fields contribute to principle scores, stakeholder utilities, or descriptive bias indicators. Because capacity is not a constraint in this example, the resulting allocation is a scoring illustration rather than an operationally feasible sourcing plan.

In [ ]:
ethical_supplier_database = {
    "FairTrade_Africa": {
        "location": "Kenya", "cost": 320, "capacity": 25000,
        "quality": 0.88, "ethical_score": 0.95, "environmental_score": 0.92,
        "social_score": 0.96, "wage_multiplier": 2.5, "safety_score": 0.92,
        "working_conditions": 0.90, "carbon_intensity": 0.3,
        "renewable_energy": 0.4, "waste_recycling": 0.85,
        "biodiversity_protection": 0.88, "anti_corruption": 0.95,
        "transparency_score": 0.90, "fair_pricing": 0.88,
        "community_investment": 0.85, "local_development": 0.82,
        "education_support": 0.78, "healthcare_access": 0.80,
        "audit_frequency": 3, "stakeholder_engagement": 0.85,
        "fair_payment": 0.92, "partnership_score": 0.88,
        "capacity_development": 0.85,
    },
    "EcoTech_Europe": {
        "location": "Germany", "cost": 380, "capacity": 30000,
        "quality": 0.94, "ethical_score": 0.93, "environmental_score": 0.96,
        "social_score": 0.91, "wage_multiplier": 2.8, "safety_score": 0.95,
        "working_conditions": 0.94, "carbon_intensity": 0.2,
        "renewable_energy": 0.6, "waste_recycling": 0.92,
        "biodiversity_protection": 0.90, "anti_corruption": 0.94,
        "transparency_score": 0.92, "fair_pricing": 0.90,
        "community_investment": 0.88, "local_development": 0.85,
        "education_support": 0.82, "healthcare_access": 0.84,
        "audit_frequency": 4, "stakeholder_engagement": 0.88,
        "fair_payment": 0.94, "partnership_score": 0.90,
        "capacity_development": 0.88,
    },
    "LocalGreen_USA": {
        "location": "California, USA", "cost": 420, "capacity": 20000,
        "quality": 0.91, "ethical_score": 0.94, "environmental_score": 0.94,
        "social_score": 0.93, "wage_multiplier": 3.2, "safety_score": 0.96,
        "working_conditions": 0.95, "carbon_intensity": 0.15,
        "renewable_energy": 0.7, "waste_recycling": 0.95,
        "biodiversity_protection": 0.92, "anti_corruption": 0.96,
        "transparency_score": 0.94, "fair_pricing": 0.92,
        "community_investment": 0.92, "local_development": 0.94,
        "education_support": 0.88, "healthcare_access": 0.90,
        "audit_frequency": 4, "stakeholder_engagement": 0.92,
        "fair_payment": 0.96, "partnership_score": 0.94,
        "capacity_development": 0.90,
    },
    "Sustainable_Asia": {
        "location": "Japan", "cost": 360, "capacity": 28000,
        "quality": 0.92, "ethical_score": 0.91, "environmental_score": 0.93,
        "social_score": 0.89, "wage_multiplier": 2.6, "safety_score": 0.93,
        "working_conditions": 0.91, "carbon_intensity": 0.25,
        "renewable_energy": 0.5, "waste_recycling": 0.88,
        "biodiversity_protection": 0.86, "anti_corruption": 0.92,
        "transparency_score": 0.88, "fair_pricing": 0.86,
        "community_investment": 0.84, "local_development": 0.82,
        "education_support": 0.80, "healthcare_access": 0.82,
        "audit_frequency": 3, "stakeholder_engagement": 0.84,
        "fair_payment": 0.90, "partnership_score": 0.86,
        "capacity_development": 0.84,
    },
    "Community_LatinAmerica": {
        "location": "Brazil", "cost": 340, "capacity": 22000,
        "quality": 0.86, "ethical_score": 0.89, "environmental_score": 0.87,
        "social_score": 0.92, "wage_multiplier": 2.3, "safety_score": 0.88,
        "working_conditions": 0.86, "carbon_intensity": 0.35,
        "renewable_energy": 0.45, "waste_recycling": 0.82,
        "biodiversity_protection": 0.84, "anti_corruption": 0.88,
        "transparency_score": 0.85, "fair_pricing": 0.84,
        "community_investment": 0.90, "local_development": 0.88,
        "education_support": 0.86, "healthcare_access": 0.84,
        "audit_frequency": 2, "stakeholder_engagement": 0.86,
        "fair_payment": 0.88, "partnership_score": 0.84,
        "capacity_development": 0.82,
    },
}

print(f"Supplier records: {len(ethical_supplier_database)}")
for supplier, data in ethical_supplier_database.items():
    print(
        f"{supplier}: location={data['location']}, cost=${data['cost']}/unit, "
        f"capacity={data['capacity']:,} units, ethical score={data['ethical_score']:.0%}"
    )

## 4. Define the problem context

The context records the planning horizon, strategic priorities, requirements, and stakeholder expectations. The scenario-generation and scoring methods use the supplier records and coded rules directly; context statements that are not mapped to a threshold or score do not automatically constrain the result. This distinction matters: recording a policy requirement is not the same as enforcing it computationally.

In [ ]:
ethical_problem_context = {
    "market_condition": "values_driven",
    "time_horizon": "36 months",
    "growth_phase": "sustainable_expansion",
    "strategic_priorities": {
        "ethical_compliance": 0.4,
        "stakeholder_satisfaction": 0.3,
        "environmental_sustainability": 0.2,
        "economic_viability": 0.1,
    },
    "ethical_requirements": [
        "Zero tolerance for child labor and forced labor",
        "Minimum 2x local living wage for all workers",
        "Carbon-neutral operations by 2025",
        "80% renewable energy usage",
        "Community investment and development programs",
        "Full transparency and regular audits",
    ],
    "stakeholder_expectations": {
        "customers": "High ethical standards and sustainability",
        "investors": "Long-term value creation with ethical practices",
        "employees": "Fair treatment and safe working conditions",
        "communities": "Positive local impact and development",
        "regulators": "Compliance with applicable ethical and legal standards",
    },
}

print("Planning horizon:", ethical_problem_context["time_horizon"])
print("Growth phase:", ethical_problem_context["growth_phase"])
print("Recorded requirements:")
for requirement in ethical_problem_context["ethical_requirements"]:
    print(" -", requirement)

## 5. Screen suppliers, generate scenarios, and select a candidate

A scenario is a candidate supplier subset together with product allocations. The code samples 20 subsets from suppliers that pass the screening rules. For each subset, the allocation rule distributes each product's full demand in proportion to the suppliers' combined scores. The model then evaluates every generated scenario and selects the one with the largest combined score.

The number 20 is the scenario count used by this example, not a guarantee that every possible supplier subset has been examined. Consequently, the selected candidate is the best among the generated scenarios, not necessarily the best among all theoretically possible sourcing decisions.

In [ ]:
print("=== ETHICAL SOURCING SCENARIO ANALYSIS ===")

eligible_suppliers = ethical_ai._filter_ethical_suppliers(ethical_supplier_database)
print(
    f"Suppliers passing coded screening rules: "
    f"{len(eligible_suppliers)} of {len(ethical_supplier_database)}"
)
for supplier in eligible_suppliers:
    print(" -", supplier)

ethical_scenarios = ethical_ai._generate_ethical_scenarios(
    ethical_problem_context, ethical_supplier_database
)
evaluated_scenarios = [
    ethical_ai._evaluate_ethical_scenario(scenario, ethical_supplier_database)
    for scenario in ethical_scenarios
]

print(f"\nCandidate scenarios generated and evaluated: {len(evaluated_scenarios)}")
for scenario in evaluated_scenarios[:5]:
    print(
        f"Scenario {scenario['id']}: suppliers={len(scenario['suppliers'])}, "
        f"constitutional={scenario['constitutional_score']:.1%}, "
        f"stakeholder={scenario['stakeholder_score']:.1%}, "
        f"combined={scenario['ethical_score']:.1%}"
    )

best_ethical_scenario = max(
    evaluated_scenarios, key=lambda scenario: scenario["ethical_score"]
)

print("\nHighest-scoring generated scenario")
print("Scenario ID:", best_ethical_scenario["id"])
print(f"Constitutional score: {best_ethical_scenario['constitutional_score']:.1%}")
print(f"Stakeholder score: {best_ethical_scenario['stakeholder_score']:.1%}")
print(f"Combined ethical score: {best_ethical_scenario['ethical_score']:.1%}")

for product, allocation in best_ethical_scenario["allocation"].items():
    demand = sum(allocation.values())
    print(f"\n{product} allocation (demand {demand:,.0f} units):")
    for supplier, quantity in allocation.items():
        print(f"  {supplier}: {quantity:,.0f} units ({quantity / demand:.1%})")
    assert np.isclose(demand, {"Battery_Tech": 12000, "Solar_Panel": 8000, "Wind_Turbine": 2000}[product])

## 6. Generate a transparency report

A useful report distinguishes the component scores from their weights and states how the overall score is constructed. The report below is generated from the selected scenario's actual computed values. Its recommendations are qualitative model notes, not claims that a real-world audit or mitigation action has taken place.

In [ ]:
transparency_report = ethical_ai.transparency_reporter.generate_report(
    best_ethical_scenario,
    ethical_ai.constitutional_principles,
    ethical_ai.stakeholder_registry,
)
print("Transparency report date:", transparency_report["report_date"])
print("Scenario ID:", transparency_report["scenario_id"])

for section_name in (
    "executive_summary",
    "constitutional_compliance",
    "stakeholder_impact",
    "ethical_performance",
):
    section = transparency_report["sections"][section_name]
    print(f"\n{section['title']}\n{'-' * len(section['title'])}")
    print(section["content"])

## 7. Examine bias indicators

The detector computes five descriptive indicators. Geographic concentration, variation in supplier cost, and variation in supplier capacity are calculated from the selected suppliers. Relationship and cognitive indicators are simulated because this example contains no evidence from which to estimate them. The threshold of 0.3 is a modeling choice, not a validated fairness standard.

The returned mitigation strategies are review prompts. They do not alter the scenario or its score; therefore, the analysis should not be interpreted as a measured reduction in bias.

In [ ]:
bias_analysis = ethical_ai.bias_detector.analyze_biases(best_ethical_scenario)
print(f"Overall mean indicator score: {bias_analysis['overall_bias_risk']:.1%}")
print("Indicator scores (threshold for a review flag: 30%):")
for bias_type, score in bias_analysis["bias_scores"].items():
    status = "Review flag" if score > 0.3 else "Below threshold"
    print(f"  {bias_type.replace('_', ' ').title()}: {score:.1%} — {status}")

print("\nReview prompts:")
if bias_analysis["mitigation_strategies"]:
    for strategy in bias_analysis["mitigation_strategies"]:
        print(" -", strategy)
else:
    print("No indicator exceeded the chosen threshold.")

print("\nReport note:")
print(transparency_report["sections"]["bias_analysis"]["content"])

## 8. Calculate interpretable summary metrics

The following functions retain the example's business-impact, long-term-value, and risk-reduction ideas while making their assumptions explicit. The conventional unit cost of $250 is a comparison benchmark used only for the cost-premium calculation. Long-term value and risk reduction are illustrative proxies based on declared formulas; they are not measured financial forecasts or causal estimates.

The cost premium is

$$
\text{premium}=\frac{\text{scenario average unit cost}-250}{250}.
$$

The value proxy is \(15\,\text{million}\times E_c\times H_c\), and the risk-reduction proxy is \(0.25\times E_c\times 0.5\). These formulas are illustrative assumptions, not outputs validated against real business outcomes.

In [ ]:
def calculate_business_impact(scenario, supplier_db):
    """Calculate example cost and score indicators from allocated quantities."""
    total_quantity = 0.0
    total_cost = 0.0
    for product_allocation in scenario["allocation"].values():
        for supplier, quantity in product_allocation.items():
            total_quantity += quantity
            total_cost += supplier_db[supplier]["cost"] * quantity
    if total_quantity <= 0:
        raise ValueError("The scenario must allocate a positive total quantity.")
    average_cost = total_cost / total_quantity
    benchmark_cost = 250.0
    return {
        "average_unit_cost": average_cost,
        "cost_premium": (average_cost - benchmark_cost) / benchmark_cost,
        "brand_reputation": scenario["brand_score"],
        "customer_loyalty": scenario["stakeholder_utilities"]["customers"],
        "market_share": None,
        "roi": None,
    }


def calculate_long_term_value(scenario, supplier_db):
    """Return an illustrative value proxy in millions of dollars."""
    base_value_millions = 15.0
    return base_value_millions * scenario["ethical_score"] * scenario["stakeholder_score"]


def calculate_risk_reduction(scenario, supplier_db):
    """Return an illustrative risk-reduction proxy, not an empirical estimate."""
    base_risk = 0.25
    return base_risk * scenario["ethical_score"] * 0.5


constitutional_compliance = ethical_ai._calculate_compliance(best_ethical_scenario)
stakeholder_satisfaction = ethical_ai._calculate_stakeholder_satisfaction(best_ethical_scenario)
business_impact = calculate_business_impact(best_ethical_scenario, ethical_supplier_database)
long_term_value = calculate_long_term_value(best_ethical_scenario, ethical_supplier_database)
risk_reduction = calculate_risk_reduction(best_ethical_scenario, ethical_supplier_database)

assert np.isclose(
    best_ethical_scenario["ethical_score"],
    0.6 * constitutional_compliance + 0.4 * stakeholder_satisfaction,
)

print("Summary metrics for the selected scenario")
print(f"Constitutional compliance score: {constitutional_compliance:.1%}")
print(f"Stakeholder satisfaction score: {stakeholder_satisfaction:.1%}")
print(f"Combined ethical score: {best_ethical_scenario['ethical_score']:.1%}")
print(f"Average unit cost: ${business_impact['average_unit_cost']:.2f}")
print(f"Cost premium over the $250 benchmark: {business_impact['cost_premium']:.1%}")
print(f"Illustrative long-term value proxy: ${long_term_value:.2f} million")
print(f"Illustrative risk-reduction proxy: {risk_reduction:.1%}")
print("Market-share growth and ROI are not estimated by this model.")
print("\nRecommendations")
print(transparency_report["sections"]["recommendations"]["content"])

## 9. Visualize scenario scores and indicators

The first figure compares principle scores, stakeholder utilities, ethical and economic indicators across evaluated scenarios, and the bias indicators for the selected scenario. The economic indicator is defined here as \(1/(1+\text{average unit cost}/250)\), so it is a bounded transformation of the allocation's average unit cost rather than a separate optimization objective. The second figure visualizes the selected scenario's score components and cost premium; it does not present an invented performance timeline or unsupported before-and-after business effects.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Ethical Sourcing Scenario Analysis", fontsize=16, fontweight="bold")

principles = list(best_ethical_scenario["constitutional_scores"])
principle_scores = [best_ethical_scenario["constitutional_scores"][name] for name in principles]
angles = np.linspace(0, 2 * np.pi, len(principles), endpoint=False)
closed_angles = np.concatenate([angles, [angles[0]]])
closed_scores = principle_scores + [principle_scores[0]]
axes[0, 0].plot(closed_angles, closed_scores, "o-", linewidth=2)
axes[0, 0].fill(closed_angles, closed_scores, alpha=0.2)
axes[0, 0].set_xticks(angles)
axes[0, 0].set_xticklabels([name.replace("_", "\n").title() for name in principles])
axes[0, 0].set_ylim(0, 1)
axes[0, 0].set_title("Constitutional Principle Scores")
axes[0, 0].set_ylabel("Score (0–1)")
axes[0, 0].grid(True, alpha=0.3)

stakeholders = list(best_ethical_scenario["stakeholder_utilities"])
utilities = [best_ethical_scenario["stakeholder_utilities"][name] for name in stakeholders]
axes[0, 1].bar(stakeholders, utilities, color=sns.color_palette("husl", len(stakeholders)))
axes[0, 1].set_ylabel("Utility score (0–1)")
axes[0, 1].set_title("Stakeholder Utility Scores")
axes[0, 1].set_ylim(0, 1)
axes[0, 1].tick_params(axis="x", rotation=35)
axes[0, 1].grid(True, axis="y", alpha=0.3)

scenario_cost_scores = []
for scenario in evaluated_scenarios:
    average_cost = scenario["total_cost"] / scenario["total_quantity"]
    scenario_cost_scores.append(1.0 / (1.0 + average_cost / 250.0))
ethical_scores = [scenario["ethical_score"] for scenario in evaluated_scenarios]
scatter = axes[1, 0].scatter(
    ethical_scores, scenario_cost_scores, c=ethical_scores, cmap="viridis", s=70, alpha=0.8
)
axes[1, 0].scatter(
    [best_ethical_scenario["ethical_score"]],
    [1.0 / (1.0 + business_impact["average_unit_cost"] / 250.0)],
    marker="*", s=220, color="red", edgecolor="black", label="Selected scenario",
)
axes[1, 0].set_xlabel("Combined ethical score")
axes[1, 0].set_ylabel("Transformed cost indicator")
axes[1, 0].set_title("Ethical Score and Cost Indicator")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
fig.colorbar(scatter, ax=axes[1, 0], label="Combined ethical score")

bias_names = list(bias_analysis["bias_scores"])
bias_values = [bias_analysis["bias_scores"][name] for name in bias_names]
bias_colors = ["firebrick" if value > 0.3 else "seagreen" for value in bias_values]
axes[1, 1].bar(bias_names, bias_values, color=bias_colors, alpha=0.8)
axes[1, 1].axhline(0.3, color="black", linestyle="--", label="Review threshold")
axes[1, 1].set_ylabel("Indicator score")
axes[1, 1].set_title("Bias Indicators")
axes[1, 1].tick_params(axis="x", rotation=35)
axes[1, 1].legend()
axes[1, 1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
score_names = ["Constitutional", "Stakeholder", "Combined ethical"]
score_values = [constitutional_compliance, stakeholder_satisfaction, best_ethical_scenario["ethical_score"]]
ax.bar(score_names, score_values, color=["#4E79A7", "#59A14F", "#F28E2B"])
ax.set_ylim(0, 1)
ax.set_ylabel("Score (0–1)")
ax.set_title("Selected Scenario: Weighted Score Components")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Interpretation and responsible use

## Ethical principles and stakeholder trade-offs

The framework makes declared values visible in the scoring process. Constitutional principle weights determine how the five principle scores contribute to \(A_c\); stakeholder weights determine how the six utility scores contribute to \(H_c\). The combined score then gives constitutional compliance a weight of 0.6 and stakeholder utility a weight of 0.4. These weights are normative choices: changing them can change scenario rankings, so they require transparent justification and stakeholder review.

## What the generated result means

The selected scenario is the highest-scoring member of the 20 generated candidates under the coded screening, allocation, scoring, and weighting rules. It is not proof of ethical compliance in the real world, a globally optimal sourcing plan, or evidence that every affected stakeholder is satisfied. Supplier scores and assumptions require verification, and the allocation should be checked against capacity, legal requirements, product compatibility, logistics, and other operational constraints before practical use.

## Bias indicators, transparency, and accountability

Calculated concentration and variation indicators describe properties of the selected supplier set. The simulated relationship and cognitive indicators are random demonstrations and cannot establish the presence or absence of real bias. Threshold flags identify items for review; they do not constitute a fairness certification. Likewise, a generated report supports traceability of the model's inputs and calculations but does not replace an audit trail of evidence, human decisions, or supplier engagement.

## Appropriate use

This framework is most useful as a teaching example or as a starting point for structured discussion when an organization wants ethical and stakeholder considerations to be explicit in sourcing analysis. A real deployment would need validated evidence, clearly operationalized requirements, stakeholder participation, and suitable operational constraints. Additional constraints or broader supplier data can be incorporated while retaining the same candidate-generation and weighted-scoring sequence.